# Driven-Lorenz pipeline (demo)

**This notebook is a pipeline demonstration**, not the canonical source for the published figures.  For paper-ready PNG / PDF regeneration of every panel of Fig. 2 and Supp. Fig. 1, use `figures/figure_2B.py`, `figures/figure_2C.py`, `figures/figure_2D.py`, `figures/figure_2E.py`, and `figures/supp_figure_1.py`, or run `python figures/run_all.py` from the repository root.

What this notebook does: drives the Lorenz system from a double-well Metropolis particle, runs the multi-timescale wavelet pipeline, contrasts the resulting $\phi_2$ against a fixed-timescale baseline, and demonstrates the $\beta$ sweep that produces Fig. 2D and 2E.

In [ ]:
# --- Colab / local setup ---
# If you're running on Colab, uncomment the next two lines to install the
# pip dependencies that aren't in the default Colab image (and to clone
# this repo for the helper modules + data).
#
# !pip install pygpcca powerlaw umap-learn
# !git clone https://github.com/<your-org>/slowmode.git
# %cd slowmode

import os, sys, pickle, time
import numpy as np
import matplotlib.pyplot as plt

# Ensure slowmode/ is on the path.
sys.path.insert(0, os.path.abspath('.'))

import pipeline as pp
import gpcca_utils as gu
import figures as fg

fg.setup_style()
os.makedirs('outputs', exist_ok=True)


In [ ]:
# --- imports specific to this notebook ---
import lorenz_simulation as ls
from sklearn.decomposition import PCA


## 1. Simulate the driven Lorenz at a single $\beta$

We start with $\beta = 0.5$, the value used for Fig. 2A–C and Fig. S1A–C.
At this $\beta$ the mean dwell time per well is $\approx 60$ s (~6000
frames at 100 Hz).

The simulation runs `solve_ivp` with rtol/atol $10^{-8}$, then discards a
200 s transient.


In [ ]:
beta = 0.50
t, xyz, h = ls.simulate(beta=beta, T=2200.0, discard=200.0, fs=100.0,
                         seed=0)
print(f't.shape = {t.shape}    xyz.shape = {xyz.shape}')
print(f'mean dwell time = {ls.mean_dwell_time(h, fs=100.0):.1f} s')


## 2. Run the multi-timescale pipeline

The pipeline applied to the Lorenz $(x, y, z)$ trajectory:

1. wavelet decomposition of each of the 3 channels (25 bands, 0.5–25 Hz);
2. PCA on the resulting amplitudes, keeping components above a phase-shuffle
   threshold;
3. delay embedding with $d = 7$ frames;
4. $k$-means partition into $N = 1300$ clusters;
5. row-stochastic transition matrix $T(\tau)$ at the $\beta$-specific
   working lag (per-$\beta$ values reported in Methods, Sec. "Parameters
   for the Lorenz analysis").

We use $N = 200$ here for the single-$\beta$ demo so the notebook completes
quickly; the published analysis uses $N = 1300$.


In [ ]:
fs = 100.0
n_freqs = 25; fmin, fmax = 0.5, 25.0
d_multi = 7
N = 200          # demo value; paper uses 1300
lag_multi = 309  # 3.09 s, per-beta=0.5 working lag from main.tex line 347

amps, freqs = pp.morlet_wavelet_amplitudes(xyz, fs=fs, fmin=fmin, fmax=fmax,
                                            n_freqs=n_freqs)
proj, n_kept, eigvals_pca, thresh = pp.pca_with_shuffle_threshold(
    amps, n_shuffles=10, max_keep=20, seed=0)
print(f'wavelet amps shape = {amps.shape}')
print(f'PCA kept {n_kept} components above shuffle threshold')

X_embed = pp.delay_embed(proj[:, :n_kept], d=d_multi, tau=1)
print(f'delay-embedded shape = {X_embed.shape}')

states = pp.kmeans_partition(X_embed, N=N, n_init=20, seed=0)
print(f'partitioned into N = {N} clusters')

T_multi = pp.make_transition_matrix(states, lag=lag_multi, n_states=N)
pi_multi = pp.stationary_distribution(T_multi)
evals_multi, evecs_multi = pp.leading_eigvecs(T_multi, k=10)
print(f'leading |lambda_k| = {np.abs(evals_multi)[:5].round(4)}')


## 3. Fixed-timescale comparison

For the Fig. 2C/D negative control we run the same pipeline without the
wavelet step, embedding the raw $(x, y, z)$ trajectory at $d = 8$ frames.


In [ ]:
d_fixed = 8
lag_fixed = lag_multi  # same physical lag for a fair comparison
X_embed_fixed = pp.delay_embed(xyz, d=d_fixed, tau=1)
states_fixed = pp.kmeans_partition(X_embed_fixed, N=N, n_init=20, seed=0)
T_fixed = pp.make_transition_matrix(states_fixed, lag=lag_fixed, n_states=N)
evals_fixed, evecs_fixed = pp.leading_eigvecs(T_fixed, k=10)
print(f'fixed-timescale |lambda_k| = {np.abs(evals_fixed)[:5].round(4)}')


## Fig. 2A — driver double-well potential and sample $h(t)$

The hidden driver $h(t)$ is sampled by Metropolis dynamics in a symmetric
double-well at inverse temperature $\beta$. Three sample paths at
$\beta \in \{0.3, 0.5, 0.7\}$ illustrate the dwell-time scaling.


In [ ]:
hh = np.linspace(-3, 3, 400)
U = ls.double_well_potential(hh)

fig, (ax_u, ax_h) = plt.subplots(2, 1, figsize=(4.0, 4.6),
                                  gridspec_kw=dict(height_ratios=[1, 1.2]))
ax_u.plot(hh, U, 'k', lw=1.4)
ax_u.set_xlabel('$h$'); ax_u.set_ylabel('$U(h)$')
ax_u.set_title('double-well driver')

# Sample three short driver traces at different beta.
for b, c in zip([0.3, 0.5, 0.7], ['#bbbbbb', '#444444', '#dd2222']):
    h_b = ls.metropolis_double_well(N=20000, beta=b, seed=1)
    ax_h.plot(np.arange(20000) / 100.0, h_b, color=c, lw=0.4,
              label=fr'$\beta={b}$')
ax_h.set_xlabel('time (s)'); ax_h.set_ylabel('$h(t)$')
ax_h.legend()
plt.tight_layout()
fg.save_panel(fig, 'fig2A_driver_double_well')
plt.show()


## Fig. 2B — Lorenz attractor projections colored by $\phi_2$

We render the three orthogonal projections (xy, yz, xz) of the attractor
as RdBu_r per-bin density maps colored by the per-bin mean of the second
eigenvector $\phi_2$ of the multi-timescale transfer operator.  The
$\phi_2$-coloring cleanly separates the two metastable lobes; this is most
visible in the $(z, x)$ projection.

Each frame is mapped to its cluster's $\phi_2$ value via `states`.


In [ ]:
phi2_per_frame = evecs_multi[:, 0].real[states]
xyz_view = xyz[: len(states)]   # drop the last (d-1)*tau frames lost to embedding

fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.4))
labels = [('x', 'y'), ('y', 'z'), ('x', 'z')]
data_pairs = [(xyz_view[:, 0], xyz_view[:, 1]),
              (xyz_view[:, 1], xyz_view[:, 2]),
              (xyz_view[:, 0], xyz_view[:, 2])]
for ax, (lx, ly), (xv, yv) in zip(axes, labels, data_pairs):
    fg.plot_density_map(ax, xv, yv, phi2_per_frame, n_bins=80, min_count=10)
    ax.set_title(f'({lx}, {ly})')
fig.suptitle(r'Lorenz attractor colored by $\phi_2$ (multi-timescale)',
             y=1.02)
plt.tight_layout()
fg.save_panel(fig, 'fig2B_attractor_phi2')
plt.show()


## Fig. 2C — time series: $h(t)$ vs $\phi_2$ multi vs $\phi_2$ fixed

We plot a 5-minute window with the hidden driver $h(t)$, the
$\phi_2$ projection from the multi-timescale operator, and the $\phi_2$
projection from the fixed-timescale operator constructed from the same
trajectory.  The multi-timescale projection tracks the bistable switching
of $h(t)$; the fixed-timescale projection does not.


In [ ]:
phi2_fixed_per_frame = evecs_fixed[:, 0].real[states_fixed]

# Align everything to the multi-timescale length.
T_show = 30000           # 5 minutes at 100 Hz
t_show = t[:T_show]
phi2_multi_show = phi2_per_frame[:T_show]
phi2_fixed_show = phi2_fixed_per_frame[:T_show]
h_show = h[:T_show]

# Sign-align phi_2 with sign of h, just for display.
if np.corrcoef(phi2_multi_show, h_show)[0, 1] < 0:
    phi2_multi_show = -phi2_multi_show
if np.corrcoef(phi2_fixed_show, h_show)[0, 1] < 0:
    phi2_fixed_show = -phi2_fixed_show

fig, axes = plt.subplots(3, 1, figsize=(7.6, 4.4), sharex=True)
axes[0].plot(t_show, phi2_fixed_show, color='0.45', lw=0.5)
axes[0].set_ylabel(r'$\phi_2$ (fixed)')
axes[1].plot(t_show, phi2_multi_show, color='C0', lw=0.5)
axes[1].set_ylabel(r'$\phi_2$ (multi)')
axes[2].plot(t_show, h_show, color='k', lw=0.5)
axes[2].set_xlabel('time (s)'); axes[2].set_ylabel('$h(t)$')
plt.tight_layout()
fg.save_panel(fig, 'fig2C_time_series')
plt.show()


## $\beta$ sweep (Fig. 2D / Fig. S1D)

Recipe matches the manuscript's `compute_lorenz_corr_seeds.py`:

- **wavelets on the x-channel only** (no PCA on amplitudes),
- **delay-embed `np.abs(amps)` directly** at $d = 7$ (multi) and the raw
  $x$-trajectory at $d = 8$ (fixed),
- **per-seed characteristic-timescale lag** $\tau = \mathrm{round}(t_c)$
  with $t_c = -1/\log|\lambda_2|$ from a lag-1 transition matrix,
- **eigenvector picked by sorting `eigvals.real` descending**, taking
  index 1 (skip $\lambda = 1$ at index 0),
- **Savitzky-Golay smoothing** (window 150, polyorder 5) of $\phi_2(t)$
  before correlating with $h(t)$,
- **simulation length 18 000 s** so the long-dwell tail at $\beta \geq
  0.65$ actually contains transitions.

The default below uses 6 000 s × 3 seeds (~30 min) which already separates
the multi/fixed curves cleanly across $\beta \leq 0.57$.  Set
`T_sim = 18000` and `n_seeds = 5` for the full reproduction.


In [ ]:
from sklearn.cluster import MiniBatchKMeans

# --- Beta sweep configuration (paper-matched recipe) ---
betas   = np.array([0.35, 0.42, 0.50, 0.57, 0.65, 0.73, 0.80])
n_seeds = 5                # number of initial seeds 
T_sim   = 6000.0           # simulation time in seconds (paper uses 18000)
discard = 200.0            # burn-in time to discard at the start of each simulation
fs      = 100.0            # sampling frequency in Hz 
nchan   = 25               # number of wavelet frequencies to use as input features to PCA
fmin_, fmax_ = 0.5, 25.0   # frequency range for wavelet features in Hz
N_MULTI = 1300             # number of clusters for multi-timescale analysis (paper-matched)
N_FIXED = 1300             # number of clusters for fixed-timescale analysis (paper-matched)
D_MULTI = 7                # delay embedding dimension for multi-timescale analysis
D_FIXED = 8                # delay embedding dimension for fixed-timescale analysis

def cluster_seeded(E, N_clusters, seed):
    km = MiniBatchKMeans(n_clusters=N_clusters, init='random', n_init=20,
                         batch_size=N_clusters * 5,
                         random_state=int(seed)).fit(E)
    return km.labels_.astype(int)

results = {b: {'multi': [], 'fixed': [], 'dwell': [],
               'lag_multi': [], 'lag_fixed': []} for b in betas}

for b in betas:
    for seed in range(n_seeds):
        print(f'  beta={b:.2f}, seed={seed}')
        t_b, xyz_b, h_b = ls.simulate(beta=float(b), T=T_sim, discard=discard,
                                       fs=fs, seed=seed)
        results[b]['dwell'].append(ls.mean_dwell_time(h_b, fs=fs))

        # ---- Multi-timescale: wavelets on x-channel only, no PCA. ----
        x_only = xyz_b[:, 0]
        amps_mt, _ = pp.morlet_wavelet_amplitudes(x_only, fs=fs, fmin=fmin_,
                                                   fmax=fmax_, n_freqs=nchan)
        amps_mt = np.abs(amps_mt).astype(np.float32)
        E_mt = pp.delay_embed(amps_mt, d=D_MULTI, tau=1).astype(np.float32)
        s_mt = cluster_seeded(E_mt, N_MULTI, seed)
        r_mt, lag_mt, _ = pp.phi2_correlation_preet(s_mt, h_b[D_MULTI - 1:],
                                                     fs=fs)
        results[b]['multi'].append(r_mt)
        results[b]['lag_multi'].append(lag_mt)
        del amps_mt, E_mt, s_mt

        # ---- Fixed-timescale: raw x trajectory, no wavelets. ----
        x_reshape = x_only.reshape(-1, 1).astype(np.float32)
        E_ft = pp.delay_embed(x_reshape, d=D_FIXED, tau=1).astype(np.float32)
        s_ft = cluster_seeded(E_ft, N_FIXED, seed)
        r_ft, lag_ft, _ = pp.phi2_correlation_preet(s_ft, h_b[D_FIXED - 1:],
                                                     fs=fs)
        results[b]['fixed'].append(r_ft)
        results[b]['lag_fixed'].append(lag_ft)
        del x_reshape, E_ft, s_ft

    print(f'beta={b:.2f}  dwell={np.nanmean(results[b]["dwell"]):.1f}s  '
          f'|r|_multi={np.nanmean(results[b]["multi"]):.2f}  '
          f'|r|_fixed={np.nanmean(results[b]["fixed"]):.2f}')

np.savez('outputs/lorenz_beta_sweep.npz',
         betas=np.array(betas),
         dwell=np.array([np.nanmean(results[b]['dwell']) for b in betas]),
         r_multi=np.array([results[b]['multi'] for b in betas]),
         r_fixed=np.array([results[b]['fixed'] for b in betas]),
         lag_multi=np.array([results[b]['lag_multi'] for b in betas]),
         lag_fixed=np.array([results[b]['lag_fixed'] for b in betas]))


## Fig. 2D — Pearson $|r(\phi_2, h)|$ vs mean dwell time


In [ ]:
data = np.load('outputs/lorenz_beta_sweep.npz')
mean_r_multi = data['r_multi'].mean(axis=1)
sem_r_multi  = data['r_multi'].std(axis=1) / np.sqrt(data['r_multi'].shape[1])
mean_r_fixed = data['r_fixed'].mean(axis=1)
sem_r_fixed  = data['r_fixed'].std(axis=1) / np.sqrt(data['r_fixed'].shape[1])

fig, ax = plt.subplots(figsize=(4.6, 3.2))
ax.errorbar(data['dwell'], mean_r_multi, yerr=sem_r_multi, marker='o',
             color='C0', label='multi-timescale')
ax.errorbar(data['dwell'], mean_r_fixed, yerr=sem_r_fixed, marker='s',
             color='0.45', label='fixed-timescale')
ax.set_xscale('log')
ax.set_xlabel('mean dwell time (s)')
ax.set_ylabel(r'$|r(\phi_2, h)|$')
ax.set_ylim(0, 1.02)
ax.legend()
plt.tight_layout()
fg.save_panel(fig, 'fig2D_correlation_vs_dwell')
plt.show()


## Fig. S1A — Cao's $E_1(d)$ saturation

Cao's $E_1(d)$ statistic at the $\beta = 0.5$ trajectory.  Saturation at
$d = 7$ supports the embedding-dimension choice; the fixed-timescale (no
wavelet) version saturates one frame later.


In [ ]:
E1_multi = pp.cao_e1(proj[:, :n_kept], max_d=15, tau=1)
E1_fixed = pp.cao_e1(xyz, max_d=15, tau=1)
fig, ax = plt.subplots(figsize=(4.0, 2.8))
ax.plot(np.arange(2, len(E1_multi) + 2), E1_multi, 'o-', color='C0',
         label='multi (wavelet PCs)')
ax.plot(np.arange(2, len(E1_fixed) + 2), E1_fixed, 's-', color='0.45',
         label='fixed (raw xyz)')
ax.axhline(1.0, ls=':', color='0.6')
ax.set_xlabel(r'$d$'); ax.set_ylabel(r'$E_1(d)$'); ax.legend()
plt.tight_layout()
fg.save_panel(fig, 'figS1A_cao_E1')
plt.show()


## Fig. S1B — Entropy gap $\Delta H(N)$

The entropy gap criterion picks $N$ as the value at which the per-step
Markov entropy difference between the true sequence and a Shannon-shuffled
surrogate is largest.  Reduce `N_values` for a fast demo.


In [ ]:
N_values = [50, 100, 200, 400, 800]
Ns, H, Hs = pp.entropy_gap(X_embed, N_values, lag=lag_multi, framerate=fs,
                            seed=0, n_init=5, verbose=True)

fig, ax = plt.subplots(figsize=(4.0, 2.8))
ax.plot(Ns, Hs - H, 'o-', color='C0')
ax.set_xscale('log')
ax.set_xlabel(r'$N$ (clusters)'); ax.set_ylabel(r'$\Delta H(N)$ (bits/s)')
plt.tight_layout()
fg.save_panel(fig, 'figS1B_entropy_gap')
plt.show()


## Fig. S1C — Eigenvalue spectra: multi vs fixed


In [ ]:
fig, ax = plt.subplots(figsize=(4.0, 2.8))
ax.scatter(np.arange(1, 11) - 0.12, np.abs(evals_multi)[:10], s=36,
           color='C0', edgecolors='none', label='multi', zorder=3)
ax.scatter(np.arange(1, 11) + 0.12, np.abs(evals_fixed)[:10], s=36,
           color='0.45', edgecolors='none', label='fixed', zorder=3)
ax.set_xlabel('$k$'); ax.set_ylabel(r'$|\\lambda_k|$')
ax.set_xticks(np.arange(1, 11))
ax.set_ylim(0, 1.02); ax.legend()
plt.tight_layout()
fg.save_panel(fig, 'figS1C_eigenvalue_spectra')
plt.show()


## Fig. S1D — Mean dwell time vs $\beta$


In [ ]:
fig, ax = plt.subplots(figsize=(4.0, 2.8))
ax.plot(data['betas'], data['dwell'], 'o-', color='k')
ax.set_yscale('log')
ax.set_xlabel(r'$\beta$'); ax.set_ylabel('mean dwell time (s)')
plt.tight_layout()
fg.save_panel(fig, 'figS1D_dwell_vs_beta')
plt.show()


---

All Lorenz panels are now saved as PNG/PDF under `outputs/`. The next two
notebooks (`worms.ipynb` and `flies.ipynb`) apply the same pipeline to the
two biological systems analyzed in the paper.
